# Wrong-way detection on a Colab GPU

Runs `pipeline.py` (RF-DETR + ByteTrack + `WrongWayDetector`) against a
dashcam video.

Works either in the browser at colab.research.google.com, or in VS Code
through the official Google Colab extension. Either way the code
executes on a **remote Google machine**, which cannot see your local
disk -- so the cells below pull the code from GitHub and the video from
Drive rather than assuming anything is already there.

Pick a GPU runtime before running. Free tier gives a T4; a Colab Pro
subscription unlocks faster ones (L4, A100) and the extension can use
those too.

**Read the class-id table in the run cell's output before you read any
alert.** It settles the open question in `CLAUDE.md`: whether
`VEHICLE_CLASS_IDS = {2, 3, 5, 7}` matches this RF-DETR build. If the
names beside those ids are not vehicles, every alert below them is
meaningless.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Switch the runtime to a GPU type, then rerun this cell.")

## 1. Install

Only two packages. The runtime already ships `torch` (CUDA build),
`opencv` and `numpy`; installing `requirements.txt` wholesale can replace
the CUDA torch with a CPU one and silently cost you the GPU.
`supervision` arrives as a dependency of `rfdetr`.

In [ ]:
!pip install -q rfdetr trackers

## 2. Get the code

Clones on the first run of a session, pulls on every run after that.
**Re-run this cell after every push** -- it is what carries your local
edits across to the machine that actually executes them.

In [ ]:
import os

REPO_URL = "https://github.com/Arielevi15/Crime_Traffic_Dedector.git"
REPO_DIR = "/content/Crime_Traffic_Dedector"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only
!ls

## 3. Get the video

Put your dashcam clips in a folder in Google Drive once, and every future
session sees them without another upload. Mounting opens an auth prompt
the first time.

If you would rather not use Drive, replace this cell with
`from google.colab import files; files.upload()` and accept that the
upload repeats every session.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Point this at wherever you actually put the file.
VIDEO = "/content/drive/MyDrive/dashcam/dashcam.mp4"

assert os.path.isfile(VIDEO), "Not found: {0}\nRun !ls on the folder to check the name.".format(VIDEO)
print("Video ready:", VIDEO)

## 4. Run

`limit_frames=300` keeps the first run short: long enough to produce the
class-id table and show whether tracking holds, short enough that a
misconfiguration costs seconds rather than an hour. Drop it once the
class ids are confirmed.

In [ ]:
from pipeline import run

alerts = run(
    video=VIDEO,
    output="check.mp4",
    limit_frames=300,
)
alerts

## 5. Watch the annotated result

Green box = tracked vehicle, red = alerted, orange dot = the
road-contact point the detector actually reasons about. If those dots are
not landing on the road beneath each vehicle, fix that before tuning any
threshold -- everything downstream depends on that point being right.

OpenCV writes `mp4v`, which the notebook player will not decode, so
re-encode to H.264 first. The video is inlined as base64, which is fine
for a few hundred frames; for a full clip, download it instead.

In [ ]:
from base64 import b64encode

from IPython.display import HTML

!ffmpeg -loglevel error -i check.mp4 -vcodec libx264 -y check_h264.mp4

payload = b64encode(open("check_h264.mp4", "rb").read()).decode()
HTML('<video width=720 controls><source src="data:video/mp4;base64,{0}">'.format(payload))

In [ ]:
# Longer clips: copy the result back to Drive instead of inlining it.
!cp check_h264.mp4 /content/drive/MyDrive/dashcam/

## 6. Tuning

Once the class ids are confirmed, the next open task is tuning
`DetectorConfig` against real footage. Pass one in explicitly rather than
editing the module, so the tested defaults stay intact:

```python
from wrong_way_detector import DetectorConfig

alerts = run(
    video=VIDEO,
    output="check.mp4",
    config=DetectorConfig(opposite_cos_threshold=-0.6),
)
```

Per `CLAUDE.md` principle 3, tune toward silence. A false positive
accuses an innocent driver; a false negative merely misses one.